In [1]:
import numpy as np
import pandas as pd 
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input ,Dense ,Dropout
from tensorflow.keras.optimizers import Adam


E:\zhrenal\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
np.random.seed(42)
np_sample=10000


In [3]:
data={
    #input 
    'age':np.random.randint(18,80,np_sample),
    'BMI':np.random.randint(6,28,np_sample),
    'Systolic_blood_pressure':np.random.randint(18,125,np_sample),#(mmHg)
    'Fasting_blood_sugar_level':np.random.randint(25,105,np_sample),#(mg/dl)
    'Hours_of_physical_activity_per_week':np.random.randint(0,12,np_sample),
}

df=pd.DataFrame(data) 

#regression output
df['cholesterol_level']=(
    135+1.7*df['age']+
    3.4*df['BMI']+
    0.43*df['Systolic_blood_pressure']+
    0.37*df['Fasting_blood_sugar_level']-
    3.0*df['Hours_of_physical_activity_per_week']+
    np.random.normal(0,15,np_sample
                    ))

df['cholesterol_level']=np.clip(df['cholesterol_level'],118,325)

#Classification
risk_score = (
    0.038 * df['age'] +
    0.125 * df['BMI'] +
    0.027 * df['Systolic_blood_pressure'] +
    0.02 * df['Fasting_blood_sugar_level'] -
    0.105 * df['Hours_of_physical_activity_per_week']
)
df['heart_risk'] = (risk_score > risk_score.median()).astype(int)

print(df.head())
print('\n Cholesterol_level statistics:')
print(df['cholesterol_level'].describe())


   age  BMI  Systolic_blood_pressure  Fasting_blood_sugar_level  \
0   56   17                       60                         31   
1   69   10                       51                         36   
2   46   26                      114                         62   
3   32   21                       66                         78   
4   60   18                       35                         91   

   Hours_of_physical_activity_per_week  cholesterol_level  heart_risk  
0                                   11         274.354197           0  
1                                    6         285.917375           0  
2                                    8         320.087557           1  
3                                    5         316.235698           1  
4                                    5         325.000000           1  

 Cholesterol_level statistics:
count    10000.000000
mean       299.786765
std         30.226804
min        158.562199
25%        280.738308
50%        312.305976
7

In [4]:
x=df[['age','BMI','Systolic_blood_pressure','Hours_of_physical_activity_per_week']].values
y_reg=df['cholesterol_level'].values
y_class=df['heart_risk'].values



In [5]:
x_train, x_test, y_reg_train, y_reg_test, y_class_train, y_class_test = train_test_split(
    x, y_reg, y_class, test_size=0.2, random_state=42, stratify=y_class)

In [6]:
scaler=StandardScaler()
x_scaler=scaler.fit_transform(x)

In [7]:
input_layer=Input(shape=(4,),name='input_features')
x= Dense(64,activation='relu')(input_layer)
x=Dropout(0.3)(x)
x=Dense(48,activation='relu')(x)
x=Dense(24,activation='relu')(x)


In [8]:
#output1: reg
cholesterol_out = Dense(1, activation='linear', name='cholesterol_level')(x)
#output2:class
risk_out=Dense(1,activation='sigmoid' ,name='heart_risk')(x)

In [9]:
model = Model(inputs=input_layer, outputs=[cholesterol_out, risk_out]) 

In [10]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={'cholesterol_level': 'mse', 'heart_risk': 'binary_crossentropy'},
    metrics={'cholesterol_level': ['mae'], 'heart_risk': ['accuracy']}
)

In [11]:
history = model.fit(
    x_scaler,
    {'cholesterol_level': y_reg,'heart_risk': y_class},
    epochs=80,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/80
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - cholesterol_level_loss: 74497.3594 - cholesterol_level_mae: 268.2576 - heart_risk_accuracy: 0.5082 - heart_risk_loss: 0.7168 - loss: 74498.1016 - val_cholesterol_level_loss: 25268.2871 - val_cholesterol_level_mae: 153.8717 - val_heart_risk_accuracy: 0.6305 - val_heart_risk_loss: 0.6655 - val_loss: 25364.3320
Epoch 2/80
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - cholesterol_level_loss: 6116.7310 - cholesterol_level_mae: 62.1464 - heart_risk_accuracy: 0.5757 - heart_risk_loss: 0.6750 - loss: 6117.4077 - val_cholesterol_level_loss: 2066.0891 - val_cholesterol_level_mae: 35.6335 - val_heart_risk_accuracy: 0.5575 - val_heart_risk_loss: 0.6596 - val_loss: 2063.1375
Epoch 3/80
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - cholesterol_level_loss: 3234.5312 - cholesterol_level_mae: 46.1422 - heart_risk_accuracy: 0.5979 - heart_risk_loss: 0.6614 - loss: 3235.1946 - val_cholesterol_level_loss: 1680.0963 - val_cholesterol_level_mae: 32.3605 - val_